In [1]:
# =========================
# AUDIO FILE DIAGNOSTIC TOOL
# =========================
import os
import numpy as np
import librosa
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

def diagnose_audio_files(data_dir):
    """Diagnostic tool untuk memeriksa kualitas file audio"""
    data_root = Path(data_dir)
    
    print("🔍 AUDIO FILE DIAGNOSIS")
    print("=" * 60)
    
    # Check overall structure
    if not data_root.exists():
        print(f"❌ Directory tidak ditemukan: {data_dir}")
        return
    
    class_dirs = [d for d in data_root.iterdir() if d.is_dir()]
    if not class_dirs:
        print(f"❌ Tidak ada subdirectory kelas di: {data_dir}")
        return
    
    print(f"📁 Directory: {data_dir}")
    print(f"📂 Jumlah kelas: {len(class_dirs)}")
    print("=" * 60)
    
    total_files = 0
    valid_files = 0
    problematic_files = []
    
    for class_dir in class_dirs:
        print(f"\n📂 Kelas: {class_dir.name}")
        print("-" * 40)
        
        # Find audio files
        audio_files = []
        for ext in [".wav", ".mp3", ".flac", ".m4a", ".ogg"]:
            audio_files.extend(list(class_dir.glob(f"*{ext}")))
            audio_files.extend(list(class_dir.glob(f"*{ext.upper()}")))
        
        print(f"📄 File audio ditemukan: {len(audio_files)}")
        
        class_valid = 0
        for audio_file in audio_files[:10]:  # Check first 10 files per class
            total_files += 1
            print(f"  🔍 Analyzing: {audio_file.name}")
            
            try:
                # Check file size
                file_size = audio_file.stat().st_size
                print(f"    📏 Size: {file_size} bytes")
                
                if file_size < 1000:  # Less than 1KB
                    print(f"    ❌ File terlalu kecil (mungkin corrupt)")
                    problematic_files.append((audio_file, "File terlalu kecil"))
                    continue
                
                # Try to load audio
                y, sr = librosa.load(str(audio_file), sr=None, mono=True)
                
                # Basic audio analysis
                duration = len(y) / sr
                max_amplitude = np.max(np.abs(y))
                rms = np.sqrt(np.mean(y**2))
                
                print(f"    ⏱️  Duration: {duration:.2f}s")
                print(f"    🔊 Max amplitude: {max_amplitude:.4f}")
                print(f"    📊 RMS: {rms:.6f}")
                print(f"    🎵 Sample rate: {sr} Hz")
                print(f"    📈 Samples: {len(y)}")
                
                # Check audio quality
                if duration < 0.1:
                    print(f"    ❌ Audio terlalu pendek (< 0.1s)")
                    problematic_files.append((audio_file, "Audio terlalu pendek"))
                elif max_amplitude < 0.001:
                    print(f"    ❌ Audio terlalu senyap (amplitude rendah)")
                    problematic_files.append((audio_file, "Audio terlalu senyap"))
                elif rms < 0.0001:
                    print(f"    ❌ RMS terlalu rendah")
                    problematic_files.append((audio_file, "RMS rendah"))
                else:
                    print(f"    ✅ File audio VALID")
                    class_valid += 1
                    valid_files += 1
                    
            except Exception as e:
                print(f"    ❌ ERROR: {e}")
                problematic_files.append((audio_file, str(e)))
        
        print(f"  ✅ Valid files in {class_dir.name}: {class_valid}/{len(audio_files)}")
    
    print("\n" + "=" * 60)
    print("📊 DIAGNOSIS SUMMARY")
    print("=" * 60)
    print(f"📁 Total directories: {len(class_dirs)}")
    print(f"📄 Total files checked: {total_files}")
    print(f"✅ Valid files: {valid_files}")
    print(f"❌ Problematic files: {len(problematic_files)}")
    
    if problematic_files:
        print(f"\n🚨 PROBLEMATIC FILES:")
        for file, issue in problematic_files[:10]:  # Show first 10
            print(f"  ❌ {file.name}: {issue}")
        if len(problematic_files) > 10:
            print(f"  ... and {len(problematic_files) - 10} more")
    
    # Recommendations
    print(f"\n💡 RECOMMENDATIONS:")
    if valid_files == 0:
        print("1. ❌ SEMUA FILE AUDIO PROBLEMATIC!")
        print("2. Download ulang dataset atau buat recording baru")
        print("3. Pastikan format file supported (.wav, .mp3)")
        print("4. Check recording equipment dan settings")
    elif valid_files < 50:
        print("1. ⚠️  Terlalu sedikit file valid")
        print("2. Butuh lebih banyak data training")
        print("3. Consider data augmentation")
    else:
        print("1. ✅ Cukup file valid untuk training")
        print("2. Lanjutkan dengan model training")

# Run diagnosis
DATA_DIR = r"D:\web\cnn_clasification\archive\Data\genres_original"
diagnose_audio_files(DATA_DIR)

🔍 AUDIO FILE DIAGNOSIS
📁 Directory: D:\web\cnn_clasification\archive\Data\genres_original
📂 Jumlah kelas: 8

📂 Kelas: audioanjing
----------------------------------------
📄 File audio ditemukan: 50
  🔍 Analyzing: anjing1.wav
    📏 Size: 17358926 bytes
    ⏱️  Duration: 98.41s
    🔊 Max amplitude: 0.9482
    📊 RMS: 0.158815
    🎵 Sample rate: 44100 Hz
    📈 Samples: 4339712
    ✅ File audio VALID
  🔍 Analyzing: anjing13.wav
    📏 Size: 3830086 bytes
    ⏱️  Duration: 21.71s
    🔊 Max amplitude: 1.0000
    📊 RMS: 0.150794
    🎵 Sample rate: 44100 Hz
    📈 Samples: 957502
    ✅ File audio VALID
  🔍 Analyzing: anjing14.wav
    📏 Size: 3944774 bytes
    ⏱️  Duration: 22.36s
    🔊 Max amplitude: 0.7676
    📊 RMS: 0.163058
    🎵 Sample rate: 44100 Hz
    📈 Samples: 986174
    ✅ File audio VALID
  🔍 Analyzing: anjing15.wav
    📏 Size: 3617094 bytes
    ⏱️  Duration: 20.50s
    🔊 Max amplitude: 1.0000
    📊 RMS: 0.344263
    🎵 Sample rate: 44100 Hz
    📈 Samples: 904254
    ✅ File audio VALID
 